In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/sylwi/OneDrive/Dokumenty/GitHub/project-telco-churn/data/processed/telco_clean.csv")

y = df["Churn"]
X = df.drop(columns=["Churn"])

print("X:", X.shape)
print("y:", y.shape)
print(30 * "--")
print(y.value_counts(normalize=True).round(3))

X: (7043, 19)
y: (7043,)
------------------------------------------------------------
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [2]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,         
    stratify=y_temp,        
    random_state=42,
)

for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
]:
    print(f"{name:10s} X: {X_part.shape}  y: {y_part.shape}  churn: {y_part.mean():.3f}")

train      X: (4225, 19)  y: (4225,)  churn: 0.265
validation X: (1409, 19)  y: (1409,)  churn: 0.265
test       X: (1409, 19)  y: (1409,)  churn: 0.265


In [3]:
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()

print(f"Kolumny tekstowe ({len(cat_cols)}):")
print(cat_cols)
print(30 * "--")
print(f"Kolumny liczbowe ({len(num_cols)}):")
print(num_cols)

Kolumny tekstowe (15):
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
------------------------------------------------------------
Kolumny liczbowe (4):
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

scale_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
flag_cols = ["SeniorCitizen"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), scale_cols),
        ("flag", "passthrough", flag_cols),
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)  
X_val_prep = preprocessor.transform(X_val)           

print("train:", X_train_prep.shape)
print("validation:", X_val_prep.shape)

names = preprocessor.get_feature_names_out()
print(len(names))
print(names[:4])
print(names[-5:])

train: (4225, 45)
validation: (1409, 45)
45
['cat__gender_Female' 'cat__gender_Male' 'cat__Partner_No'
 'cat__Partner_Yes']
['cat__PaymentMethod_Mailed check' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'flag__SeniorCitizen']


**Model bazowy**

Musimy mieć co przebić - DummyClassifier to najgłupszy możliwy model: zawsze odpowiada „klient nie odejdzie”

Wchodzi pipleline, żeby nie było data leakage

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

dummy = Pipeline([
    ("prep", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent")),  
])

dummy.fit(X_train, y_train)         
y_val_pred = dummy.predict(X_val)    

print("accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("recall   :", round(recall_score(y_val, y_val_pred), 3))
print("precision:", round(precision_score(y_val, y_val_pred, zero_division=0), 3))
print("F1       :", round(f1_score(y_val, y_val_pred, zero_division=0), 3))
print(30 * "--")
print("Przewidziane klasy:", pd.Series(y_val_pred).value_counts().to_dict())

accuracy : 0.735
recall   : 0.0
precision: 0.0
F1       : 0.0
------------------------------------------------------------
Przewidziane klasy: {0: 1409}


Accuracy 73,5% wygląda przyzwoicie, ale model nie zrobił nic. To po prostu odsetek klientów, którzy zostali.
Recall 0%. Z klientów, którzy odeszli, nie wyłapał ani jednego.

Tu wychodzi, dlaczego accuracy nie ma sensu dla takiego modelu. 

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

logreg = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

logreg.fit(X_train, y_train)

y_val_pred = logreg.predict(X_val)                 
y_val_proba = logreg.predict_proba(X_val)[:, 1]    

print("accuracy :", round(accuracy_score(y_val, y_val_pred), 3))
print("recall   :", round(recall_score(y_val, y_val_pred), 3))
print("precision:", round(precision_score(y_val, y_val_pred), 3))
print("F1       :", round(f1_score(y_val, y_val_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y_val, y_val_proba), 3))
print("PR-AUC   :", round(average_precision_score(y_val, y_val_proba), 3))
print(30 * "--")
print("Macierz pomyłek (wiersze = prawda, kolumny = predykcja):")
print(confusion_matrix(y_val, y_val_pred))

accuracy : 0.803
recall   : 0.535
precision: 0.658
F1       : 0.59
ROC-AUC  : 0.836
PR-AUC   : 0.642
------------------------------------------------------------
Macierz pomyłek (wiersze = prawda, kolumny = predykcja):
[[931 104]
 [174 200]]


Sprawdzam poprawność:

naprawdę został (0): 931 (trafnie) ; 104 (fałszywy alarm)

naprawdę odszedł (1): 174 (przeoczony) ; 200 (trafnie wyłapany)

931 + 104 = 1035 klientów, którzy zostali,
174 + 200 = 374 klientów, którzy odeszli.

Razem 1035 + 374 = 1409, czyli cała validation. Odsetek odejść to 374 / 1409 = 26,5%, zgadza się z EDA.

## Podnosimy recall

